# AutoPBR v2 — Run Pipeline (config-driven)

Questo notebook esegue la pipeline leggendo **un unico file di configurazione**:

- `config/pipeline.yaml`

Vantaggi:
- un solo posto dove cambiare parametri (SAM3 / Nemotron / soglie / paths)
- riproducibile per colleghi e reviewer
- stesso comportamento del terminale (esecuzione via CLI)

> Prima di eseguire: installa `pyyaml` (`pip install pyyaml`) e `dotenv`, poi imposta `NVIDIA_API_KEY` in un file `.env` in questa directory (`export NVIDIA_API_KEY={your-nvidia-api-key}`).


In [2]:
from pathlib import Path
import os, sys, subprocess, shlex
import yaml

CONFIG_PATH = Path("pipeline.yaml")
REPO_ROOT = Path(".").resolve()

print("Repo:", REPO_ROOT)
print("Config:", CONFIG_PATH.resolve())

if not os.path.exists(".env"):
    with open(".env", "w") as f:
        f.writeline("export NVIDIA_API_KEY={your-nvidia-api-key}")


Repo: /datadrive/AutoPBR
Config: /datadrive/AutoPBR/pipeline.yaml


## Helper: run comandi con output live


In [3]:
from dotenv import load_dotenv
from huggingface_hub import notebook_login

load_dotenv()
notebook_login()

def run(cmd: str):
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def q(x) -> str:
    return shlex.quote(str(x))

def args_to_cli(args: dict, ctx: dict) -> str:
    parts = []
    for k, v in args.items():
        flag = f"--{k}"
        if isinstance(v, bool):
            if v:
                parts.append(flag)
            continue
        if isinstance(v, str):
            v = v.format(**ctx)
        parts.append(flag)
        parts.append(str(v))
    return " ".join(q(p) for p in parts)


## Load config + sanity checks


In [4]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing {CONFIG_PATH}. Create/commit it in the repo.")

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

photos_dir = cfg["paths"]["photos_dir"]
out_root   = cfg["paths"]["output_root"]
ctx = {"photos_dir": photos_dir, "output_root": out_root}

# Env check
missing = [k for k in cfg.get("env", {}).get("required", []) if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}. Set them before running (e.g., export NVIDIA_API_KEY=...).")
print("✅ Env OK")

# Input folder check
photos_path = REPO_ROOT / photos_dir
photos_path.mkdir(exist_ok=True, parents=True)
imgs = list(photos_path.glob("*.jpg")) + list(photos_path.glob("*.jpeg")) + list(photos_path.glob("*.png"))
print(f"📷 images in {photos_path}: {len(imgs)}")


✅ Env OK
📷 images in /datadrive/AutoPBR/photos: 1529


## SAM3 (segmentation)

Nota: i prompt nel YAML sono documentativi; se `2_sam3hi.py` non espone flag per i prompt, li ignora comunque.
Qui passiamo solo flag stabili (paths / per_building / bbox gating) *se presenti nel YAML*.


In [6]:
sam3 = cfg.get("sam3", {})
if sam3.get("enabled", True):
    script = sam3.get("script", "2_sam3hi.py")

    print(sam3)

    # pass only known scalar args if provided
    args = {}
    for k in ["out_dir", "score_thr", "mask_thr", "images_dir", "pattern"]:
        if k in sam3:
            kk = k.replace("_", "-")
            args[kk] = sam3[k]

    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ SAM3 disabled in config")


{'enabled': True, 'script': '2_sam3hi.py', 'classes': {'wall': ['building facade', 'wall', 'building front'], 'roof': ['building roof', 'roof'], 'road': ['road', 'street', 'road surface', 'asphalt road', 'driveway', 'sidewalk', 'pavement', 'footpath', 'walkway', 'pedestrian walkway']}, 'per_building': True, 'min_inst_bbox_h_wall': 70, 'min_inst_bbox_h_roof': 110, 'out_dir': '{output_root}/sam3_instances', 'images_dir': '{photos_dir}', 'pattern': '*.jpg', 'score_thr': 0.6, 'mask_thr': 0.5}

▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/2_sam3hi.py --out-dir data_output/sam3_instances --score-thr 0.6 --mask-thr 0.5 --images-dir photos --pattern '*.jpg'
🖼️  Found 1529 images in: photos
🔄 Loading SAM3...


Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 2536.01it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                       


✅ SAM3 ready | device=cuda | dtype=torch.float16

📸 Processing 1003317145127470.jpg


  0%|          | 0/1529 [00:00<?, ?it/s]

🏢 Found 6 buildings

📸 Processing 1004886326984596.jpg


...

🏢 Found 3 buildings

📸 Processing 998374918824983.jpg


100%|█████████▉| 1528/1529 [2:34:19<00:04,  4.09s/it]

🏢 Found 6 buildings


100%|██████████| 1529/1529 [2:34:23<00:00,  6.06s/it]



🎉 DONE
📄 Manifest: data_output/sam3_instances/manifest.json
🗂️  Output folder: data_output/sam3_instances


## Proxy semantic check


In [8]:
proxy = cfg.get("proxy_check", {})
if proxy.get("enabled", True):
    script = proxy.get("script", "3_proxy_semantic_check.py")
    args = {k: v for k, v in proxy.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ proxy_check disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/3_proxy_semantic_check.py --out_dir data_output/sam3_instances --manifest data_output/sam3_instances/manifest.json
🔌 device=cuda
🅰️ Proxy A (SegFormer): nvidia/segformer-b5-finetuned-ade-640-640
🅱️ Proxy B (OneFormer): shi-labs/oneformer_ade20k_swin_large


Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 9121.33it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]             
The image processor of type `OneFormerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 826/826 [00:00<00:00, 8305.27it/s, Materializing param=model.transformer_module.queries_embedder.weight]                                                       


✅ Done
📄 data_output/sam3_instances/proxy_global_iou.csv
📄 data_output/sam3_instances/proxy_per_building_iou.csv


## Nemotron structured


In [ ]:
ns = cfg.get("nemotron_structured", {})
if ns.get("enabled", True):
    script = ns.get("script", "4_nemotron_building_priors.py")
    args = {k: v for k, v in ns.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ nemotron_structured disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/4_nemotron_building_priors.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --canon_mode soft --per_building --skip_far_buildings --manifest data_output/sam3_instances/manifest.json --masked_dir data_output/masked_rgb --out_json materials_full_filtered.json --min_cov 1.0 --min_cov_roof 1.5 --min_cov_building 0.5 --min_cov_roof_building 0.5 --roof_conf_threshold 0.6 --crop_pad 24 --min_crop_side 64 --max_retries 3 --retry_backoff 0.8
🔄 Found existing output file at materials_full_filtered.json. Loading progress...
[1/1529] ⏭️ 1003317145127470 (Already processed, skipping)
[2/1529] ⏭️ 1004886326984596 (Already processed, skipping)
[3/1529] ⏭️ 1005755808085218 (Already processed, skipping)
[4/1529] ⏭️ 1006395093438916 (Already processed, skipping)
[5/1529] ⏭️ 1009719100838582 (Already processed, skipping)
[6/1529] ⏭️ 1010188449511786 (Already processed, skipping)
[7/1529] ⏭️ 1010335640916933 (Already processed, skippi

In [24]:
import os
from openai import OpenAI

# 1. Try to see what proxies your server is using right now
print("Current Proxy Settings:")
for k, v in os.environ.items():
    if 'proxy' in k.lower():
        print(f"  {k} = {v}")

api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    print("\n❌ ERROR: NVIDIA_API_KEY is not set!")
    exit()

print("\n🔄 Attempting to connect to NVIDIA API...")

try:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)
    
    # Send a tiny ping to the API
    response = client.chat.completions.create(
        model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1",
        messages=[{"role": "user", "content": "Reply with 'API is working!'"}]
    )
    print("\n✅ SUCCESS! The API replied:")
    print(response.choices[0].message.content)
    
except Exception as e:
    print(f"\n❌ CONNECTION FAILED: {type(e).__name__}")
    print(str(e))

Current Proxy Settings:

🔄 Attempting to connect to NVIDIA API...

✅ SUCCESS! The API replied:
API is working!


## Baseline full-image


In [4]:
nbaseline = cfg.get("nemotron_baseline", {})
if nbaseline.get("enabled", False):
    script = nbaseline.get("script", "5_nemotron_fullimage.py")
    args = {k: v for k, v in nbaseline.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ baseline disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/5_nemotron_fullimage.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --prompt_type two_pass_json_review --source_manifest data_output/sam3_instances/manifest.json --output_json baseline_full_image.json
🔄 Found existing output file at data_output/baseline_full_image.json. Loading progress...
[1/1529] ⏭️ image_1 (Already processed, skipping)
[2/1529] ⏭️ image_2 (Already processed, skipping)
[3/1529] ⏭️ image_3 (Already processed, skipping)
[4/1529] ⏭️ image_4 (Already processed, skipping)
[5/1529] ⏭️ image_5 (Already processed, skipping)
[6/1529] ⏭️ image_6 (Already processed, skipping)
[7/1529] ⏭️ image_7 (Already processed, skipping)
[8/1529] ⏭️ image_8 (Already processed, skipping)
[9/1529] ⏭️ image_9 (Already processed, skipping)
[10/1529] ⏭️ image_10 (Already processed, skipping)
[11/1529] ⏭️ image_11 (Already processed, skipping)
[12/1529] ⏭️ image_12 (Already processed, skipping)
[13/1529] ⏭️ image_13 (Alread

## Validation (optional)

In [ ]:
val = cfg.get("validation", {})
if val.get("enabled", False):
    script = val.get("script", "validator.py")
    args = {k: v for k, v in val.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ validation disabled in config")


## Quick check outputs


In [ ]:
out_candidates = [
    REPO_ROOT / "materials_full_filtered.json",
    REPO_ROOT / "baseline_full_image.json",
    REPO_ROOT / out_root / "sam3_instances" / "manifest.json",
    REPO_ROOT / out_root / "validation_results" / "report.csv",
]
for p in out_candidates:
    print(("✅" if p.exists() else "❌"), p)


## Annotations Visualization

In [4]:
ann = cfg.get("annotations", {})
if ann.get("enabled", False):
    script = ann.get("script", "gui_annotator.py")
    cmd = f"{sys.executable} {q(REPO_ROOT / script)}".strip()
    run(cmd)
else:
    print("⏭ annotations visualization disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/gui_annotator.py


The X11 connection broke: No error (code 0)
XIO:  fatal IO error 2 (No such file or directory) on X server "localhost:10.0"
      after 422 requests (422 known processed) with 0 events remaining.


CalledProcessError: Command '['/usr/local/miniconda3.8/envs/SasyEnv/bin/python', '/datadrive/AutoPBR/gui_annotator.py']' returned non-zero exit status 1.

## Detection Metrics Evaluation

In [11]:
met = cfg.get("metrics", {})
if met.get("enabled", False):
    script = met.get("script", "6_evaluate_metrics.py")
    cmd = f"{sys.executable} {q(REPO_ROOT / script)}".strip()
    run(cmd)
else:
    print("⏭ metrics evaluation disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/6_evaluate_metrics.py
⏳ Loading JSON files...
🔍 Evaluating Images...

📊 CLASSIFICATION METRICS (Baseline vs. Global vs. GT)

🏗️  --- WALL (Total Valid Objects: 4431) ---
  [MACRO AVERAGES]
    BASELINE -> Acc: 0.00 | Prec: 0.00 | Rec: 0.00 | F1: 0.00
    GLOBAL   -> Acc: 0.58 | Prec: 0.22 | Rec: 0.17 | F1: 0.17

  [PER-CLASS RESULTS]
    🔸 asphalt (Support: 56.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.50 | Rec: 0.04 | F1: 0.07
    🔸 brick (Support: 1649.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.56 | Rec: 0.74 | F1: 0.63
    🔸 concrete (Support: 1707.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.66 | Rec: 0.63 | F1: 0.64
    🔸 glass (Support: 23.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
    🔸 metal (Support: 16.0)
        Basel

## Material Download

In [5]:
cmd = f"{sys.executable} {q(REPO_ROOT / "download_materials.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/download_materials.py
🌐 Fetching texture list from Poly Haven API...
✅ Found 744 textures. Starting RGB download...

📂 Category: CONCRETE
   ⬇️ Downloaded RGB: anti_slip_concrete.jpg
   ⬇️ Downloaded RGB: asphalt_floor.jpg
   ⬇️ Downloaded RGB: brushed_concrete.jpg
   ⬇️ Downloaded RGB: brushed_concrete_03.jpg
   ⬇️ Downloaded RGB: brushed_concrete_2.jpg
   ⬇️ Downloaded RGB: climbing_wall_02.jpg
   ⬇️ Downloaded RGB: concrete.jpg
   ⬇️ Downloaded RGB: concrete_block_wall.jpg
   ⬇️ Downloaded RGB: concrete_block_wall_02.jpg
   ⬇️ Downloaded RGB: concrete_debris.jpg
   ⬇️ Downloaded RGB: concrete_floor.jpg
   ⬇️ Downloaded RGB: concrete_floor_01.jpg
   ⬇️ Downloaded RGB: concrete_floor_02.jpg
   ⬇️ Downloaded RGB: concrete_floor_damaged_01.jpg
   ⬇️ Downloaded RGB: concrete_floor_painted.jpg
📂 Category: ASPHALT
   ⬇️ Downloaded RGB: aerial_asphalt_01.jpg
   ⬇️ Downloaded RGB: asphalt_01.jpg
   ⬇️ Downloaded RGB: aspha

In [7]:
cmd = f"{sys.executable} {q(REPO_ROOT / "tint_material.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/tint_material.py
🎨 Generating color variations + preserving originals...

🔄 Processing CONCRETE...
🔄 Processing ASPHALT...
🔄 Processing WOOD...
🔄 Processing METAL...
🔄 Processing BRICK...
🔄 Processing PLASTER...
🔄 Processing TILE...
🔄 Processing SHINGLES...

🎉 Done! Generated 1180 colorized variations and preserved originals in 'colored_material_bank/'.


## Material Replacement

In [ ]:
# Execute script
mat = cfg.get("material_replacement", {})
if mat.get("enabled", False):
    script = mat.get("script", "8_zest_evaluator.py")
    cmd = f"{sys.executable} {q(REPO_ROOT / script)}".strip()
    run(cmd)
else:
    print("⏭ zest evaluation disabled in config")

⏭ zest evaluation disabled in config
Back to: /datadrive/AutoPBR


## Material Replacement with Natural Language

In [9]:
nmat = cfg.get("nlp_material_replacement", {})
if nmat.get("enabled", False):
    user_input = input("Enter your edit prompt (e.g., 'Change the wall of the biggest building on the right to white concrete'): ")
    script = nmat.get("script", "9_nlp_zest_evaluator.py")
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} --prompt \"{user_input}\"".strip()
    run(cmd)
else:
    print("⏭ nlp zest evaluation disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/9_nlp_zest_evaluator.py --prompt "Change the wall of the biggest building on the right to white concrete"
🧠 Asking AI to interpret prompt: 'Change the wall of the biggest building on the right to white concrete'
✅ AI Parsed Intent: {
  "target_instance": "building_00",
  "target_structure": "wall",
  "new_color": "white",
  "new_material": "concrete"
}

🔍 Retrieving reference image of 'white concrete' from Material Bank...
✅ Loaded high-quality reference: white_concrete/concrete_floor_02.jpg

⏳ Loading ZEST Models...


Loading weights: 100%|██████████| 414/414 [00:00<00:00, 9295.08it/s, Materializing param=neck.reassemble_stage.readout_projects.3.0.weight]                          
The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 520/520 [00:00<00:00, 2585.32it/s, Materializing param=visual_projection.weight]                                
CLIPVisionModelWithProjection LOAD REPORT from: models/image_encoder
Key                

🚀 All Pipelines Loaded.
  -> Running Naive ZEST Edit (No Segmentation)...


100%|██████████| 27/27 [00:09<00:00,  2.70it/s]
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/diffusers/pipelines/controlnet/pipeline_controlnet_inpaint_sd_xl.py:1131: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


  -> Running ZEST Edit...


100%|██████████| 27/27 [00:10<00:00,  2.69it/s]



✅ Done! Check the zest_nlp_results folder for:
   - 1003317145127470_building_00_naive_edited.jpg (No Mask Baseline)
   - 1003317145127470_building_00_nlp_edited.jpg (With Segmented Mask)


## Generation Metrics Evaluation

In [12]:
cmd = f"{sys.executable} {q(REPO_ROOT / "generate_benchmark_set.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/generate_benchmark_set.py
📖 Loading Ground Truth data...

⏳ Loading ZEST Models (ControlNet & IPAdapter)...


Loading weights: 100%|██████████| 414/414 [00:00<00:00, 6919.34it/s, Materializing param=neck.reassemble_stage.readout_projects.3.0.weight]                          
The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 520/520 [00:00<00:00, 2613.88it/s, Materializing param=visual_projection.weight]                                
CLIPVisionModelWithProjection LOAD REPORT from: models/image_encoder
Key                

🚀 Models Loaded.

[1/50] Editing 5487138621360009 -> Change the wall of building_00 to gray wood
    -> Generating Naive Edit...


100%|██████████| 27/27 [00:09<00:00,  2.70it/s]
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/diffusers/pipelines/controlnet/pipeline_controlnet_inpaint_sd_xl.py:1131: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


    -> Generating Proposed Edit...


...

    -> Generating Proposed Edit...


100%|██████████| 27/27 [00:10<00:00,  2.61it/s]


    -> Generating Improved Edit (Feathering + Compositing)...


100%|██████████| 27/27 [00:10<00:00,  2.61it/s]



🎉 Benchmark Generation Complete! Saved 50 pairs to ../benchmark_dataset.json


In [13]:
cmd = f"{sys.executable} {q(REPO_ROOT / "10_evaluate_generation.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/10_evaluate_generation.py
📖 Loading Benchmark Data...

⏳ Loading CLIP Model for Text-Image Alignment...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 6813.38it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Initializing FID Metrics...

🚀 Starting Evaluation...


Evaluating Images: 100%|██████████| 50/50 [00:23<00:00,  2.17it/s]



⏳ Computing final FID scores (this takes a few seconds)...

 📊 EVALUATION RESULTS: NAIVE vs PROPOSED vs IMPROVED
Metric               | Naive (Global)   | Proposed (Masked)  | Improved (Composited) 
-------------------------------------------------------------------------------------
Bg SSIM (↑)          |      0.3223      |       0.8090       |         0.9963        
Global CLIP (↑)      |      25.32       |       22.19        |         22.10         
Masked CLIP (↑)      |      24.29       |       24.90        |         24.97         
FID Realism (↓)      |      206.40      |       40.97        |         32.91         

✅ Detailed evaluation saved to generation_evaluation_report.json
